## Understand the baseline

In [31]:
from transformers import pipeline

model_name = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"

classifier = pipeline(
    "text-classification",
    model=model_name,
    tokenizer=model_name
)

texts = [
    "I love this movie, it was amazing!",
    "This was the worst experience ever.",
    "The product is okay, not great but not bad.",
    "The product is mediocre."
]

results = classifier(texts)

for text, res in zip(texts, results):
    print("\nTEXT:", text)
    print("LABEL:", res["label"])
    print("SCORE:", round(res["score"], 4))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


TEXT: I love this movie, it was amazing!
LABEL: POSITIVE
SCORE: 0.9999

TEXT: This was the worst experience ever.
LABEL: NEGATIVE
SCORE: 0.9998

TEXT: The product is okay, not great but not bad.
LABEL: POSITIVE
SCORE: 0.999

TEXT: The product is mediocre.
LABEL: NEGATIVE
SCORE: 0.9998


In [32]:
from datasets import load_dataset
from transformers import pipeline
from sklearn.metrics import accuracy_score, f1_score

In [33]:
model_name = "distilbert-base-uncased-finetuned-sst-2-english"

classifier = pipeline(
    "sentiment-analysis",
    model=model_name,
    tokenizer=model_name,
    truncation=True
)

dataset = load_dataset("imdb", split="test").shuffle(seed=42).select(range(500))
texts = dataset["text"]
labels = dataset["label"]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [34]:
batch_size = 16
preds = []

for i in range(0, len(texts), batch_size):
    batch_texts = texts[i:i+batch_size]
    outputs = classifier(batch_texts)
    preds.extend(outputs)

label_map = {"POSITIVE": 1, "NEGATIVE": 0}

In [35]:
y_pred = [label_map[p["label"]] for p in preds]
y_true = labels

acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average="binary", pos_label=1)

print("Accuracy:", round(acc, 4))
print("F1 Score:", round(f1, 4))

Accuracy: 0.89
F1 Score: 0.8852


In [36]:
for i in range(5):
    print("\nTEXT:", texts[i][:200])
    print("TRUE LABEL:", y_true[i])

    print("RAW MODEL OUTPUT:", preds[i])

    pred_label = preds[i]["label"]
    pred_score = preds[i]["score"]

    print("PRED LABEL:", pred_label)
    print("CONFIDENCE:", round(pred_score, 4))
    print("MAPPED PRED:", y_pred[i])


TEXT: <br /><br />When I unsuspectedly rented A Thousand Acres, I thought I was in for an entertaining King Lear story and of course Michelle Pfeiffer was in it, so what could go wrong?<br /><br />Very quic
TRUE LABEL: 1
RAW MODEL OUTPUT: {'label': 'POSITIVE', 'score': 0.9988757967948914}
PRED LABEL: POSITIVE
CONFIDENCE: 0.9989
MAPPED PRED: 1

TEXT: This is the latest entry in the long series of films with the French agent, O.S.S. 117 (the French answer to James Bond). The series was launched in the early 1950's, and spawned at least eight films 
TRUE LABEL: 1
RAW MODEL OUTPUT: {'label': 'POSITIVE', 'score': 0.9969831109046936}
PRED LABEL: POSITIVE
CONFIDENCE: 0.997
MAPPED PRED: 1

TEXT: This movie was so frustrating. Everything seemed energetic and I was totally prepared to have a good time. I at least thought I'd be able to stand it. But, I was wrong. First, the weird looping? It wa
TRUE LABEL: 0
RAW MODEL OUTPUT: {'label': 'NEGATIVE', 'score': 0.9972442388534546}
PRED LABEL: NEGATI

In [37]:
from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.87      0.92      0.89       254
           1       0.91      0.86      0.89       246

    accuracy                           0.89       500
   macro avg       0.89      0.89      0.89       500
weighted avg       0.89      0.89      0.89       500



# Fine Tune with Transformers and Accelerate

In [38]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

In [39]:
model_name = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [40]:
dataset = load_dataset("imdb")

# small subset for quick training
dataset["train"] = dataset["train"].shuffle(seed=42).select(range(2000))
dataset["test"] = dataset["test"].shuffle(seed=42).select(range(500))

In [41]:
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True)

dataset = dataset.map(tokenize, batched=True)
dataset = dataset.remove_columns(["text"])
dataset.set_format("torch")


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [42]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


In [59]:
#Convert objects to a mini batch. Adds paddign to ensure all samples in batch are the same length. 

In [43]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="binary")
    }

In [50]:
training_args = TrainingArguments(
    output_dir="./distilbert-imdb",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    report_to="none",
    disable_tqdm=True
)

In [60]:
# training_args = TrainingArguments(
# 
#     output_dir="./distilbert-imdb",
#     # where model checkpoints are saved
# 
#     learning_rate=2e-5,
#     # VERY important:
#     # LOW (1e-5):
#     #   → slow learning, stable
#     # HIGH (5e-5+):
#     #   → fast learning but unstable / may overfit
# 
#     per_device_train_batch_size=16,
#     # number of samples per GPU step
#     # HIGH:
#     #   → faster training but more memory needed
#     # LOW:
#     #   → slower but safer for small GPUs
# 
#     per_device_eval_batch_size=16,
#     # same idea but for evaluation
# 
#     num_train_epochs=2,
#     # how many times model sees full dataset
#     # LOW (1):
#     #   → underfitting
#     # HIGH (5+):
#     #   → overfitting risk
# 
#     eval_strategy="epoch",
#     # evaluates after every full pass of dataset
# 
#     save_strategy="epoch",
#     # saves model after each epoch
# 
#     logging_steps=50,
#     # logs training progress every 50 steps
#     # LOW → more detailed logs
#     # HIGH → less overhead
# 
#     report_to="none",
#     # disables wandb/other tracking tools
# 
#     disable_tqdm=True
#     # disables progress bar (useful in notebooks sometimes)
# )

In [51]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [61]:
'''# Create a Hugging Face Trainer object that automates the training process
trainer = Trainer(
    
    model=model,
    # the neural network model you want to train
    # e.g., DistilBERT, BERT, RoBERTa, etc.

    args=training_args,
    # training configuration object (TrainingArguments)
    # includes:
    # - learning rate
    # - batch size
    # - number of epochs
    # - logging settings
    # - saving strategy
    # - evaluation strategy

    train_dataset=dataset["train"],
    # dataset used to train the model
    # model learns patterns from this data

    eval_dataset=dataset["test"],
    # dataset used to evaluate model performance
    # helps check:
    # - accuracy on unseen data
    # - overfitting / underfitting

    data_collator=data_collator,
    # function that prepares batches of data
    # handles:
    # - padding sequences to same length
    # - converting samples into tensors
    # - organizing inputs for GPU processing

    compute_metrics=compute_metrics
    # function that calculates evaluation metrics
    # examples:
    # - accuracy
    # - F1 score
    # - precision / recall
    # used during evaluation phase
)
'''
# ================================
# WHAT TRAINER DOES INTERNALLY
# ================================

# 1. Loads model and datasets
# 2. Creates mini-batches using data_collator
# 3. Runs forward pass (prediction)
# 4. Computes loss
# 5. Runs backpropagation (updates weights)
# 6. Repeats for all batches in dataset
# 7. Evaluates model on eval_dataset periodically
# 8. Logs metrics (loss, accuracy, etc.)
# 9. Saves checkpoints based on training_args
# 10. Returns a trained model at the end

'# Create a Hugging Face Trainer object that automates the training process\ntrainer = Trainer(\n\n    model=model,\n    # the neural network model you want to train\n    # e.g., DistilBERT, BERT, RoBERTa, etc.\n\n    args=training_args,\n    # training configuration object (TrainingArguments)\n    # includes:\n    # - learning rate\n    # - batch size\n    # - number of epochs\n    # - logging settings\n    # - saving strategy\n    # - evaluation strategy\n\n    train_dataset=dataset["train"],\n    # dataset used to train the model\n    # model learns patterns from this data\n\n    eval_dataset=dataset["test"],\n    # dataset used to evaluate model performance\n    # helps check:\n    # - accuracy on unseen data\n    # - overfitting / underfitting\n\n    data_collator=data_collator,\n    # function that prepares batches of data\n    # handles:\n    # - padding sequences to same length\n    # - converting samples into tensors\n    # - organizing inputs for GPU processing\n\n    compute

In [52]:
trainer.train()

{'loss': '0.07926', 'grad_norm': '0.02559', 'learning_rate': '1.608e-05', 'epoch': '0.4'}
{'loss': '0.1491', 'grad_norm': '2.684', 'learning_rate': '1.208e-05', 'epoch': '0.8'}
{'eval_loss': '0.434', 'eval_accuracy': '0.896', 'eval_f1': '0.8898', 'eval_runtime': '21.38', 'eval_samples_per_second': '23.39', 'eval_steps_per_second': '1.497', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.08792', 'grad_norm': '0.0541', 'learning_rate': '8.08e-06', 'epoch': '1.2'}
{'loss': '0.03044', 'grad_norm': '0.02835', 'learning_rate': '4.08e-06', 'epoch': '1.6'}
{'loss': '0.07514', 'grad_norm': '0.08757', 'learning_rate': '8e-08', 'epoch': '2'}
{'eval_loss': '0.4646', 'eval_accuracy': '0.89', 'eval_f1': '0.8893', 'eval_runtime': '29.97', 'eval_samples_per_second': '16.69', 'eval_steps_per_second': '1.068', 'epoch': '2'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '625.6', 'train_samples_per_second': '6.394', 'train_steps_per_second': '0.4', 'train_loss': '0.08437', 'epoch': '2'}


TrainOutput(global_step=250, training_loss=0.08437042236328125, metrics={'train_runtime': 625.6346, 'train_samples_per_second': 6.394, 'train_steps_per_second': 0.4, 'train_loss': 0.08437042236328125, 'epoch': 2.0})

In [62]:
# Model learns:
# - adjusts weights slightly from pretrained SST-2 model
# - adapts to IMDB style reviews

In [56]:
results = trainer.evaluate()
print("\nEVAL RESULTS:", results)

{'eval_loss': '0.4646', 'eval_accuracy': '0.89', 'eval_f1': '0.8893', 'eval_runtime': '8.332', 'eval_samples_per_second': '60.01', 'eval_steps_per_second': '3.84', 'epoch': '2'}

EVAL RESULTS: {'eval_loss': 0.4645848870277405, 'eval_accuracy': 0.89, 'eval_f1': 0.8893360160965795, 'eval_runtime': 8.3323, 'eval_samples_per_second': 60.007, 'eval_steps_per_second': 3.84, 'epoch': 2.0}


In [63]:
# Outputs:
# - eval_loss
# - accuracy
# - f1


In [58]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

text = "This movie was surprisingly good and really well acted."

inputs = tokenizer(text, return_tensors="pt", truncation=True)
inputs = {k: v.to(device) for k, v in inputs.items()}

outputs = model(**inputs)
pred = outputs.logits.argmax(dim=1).item()

label_map = {0: "NEGATIVE", 1: "POSITIVE"}

print("\nTEXT:", text)
print("PREDICTION:", label_map[pred])


TEXT: This movie was surprisingly good and really well acted.
PREDICTION: POSITIVE


In [64]:
print(results)

{'eval_loss': 0.4645848870277405, 'eval_accuracy': 0.89, 'eval_f1': 0.8893360160965795, 'eval_runtime': 8.3323, 'eval_samples_per_second': 60.007, 'eval_steps_per_second': 3.84, 'epoch': 2.0}
